# Install

In [ ]:
pip install git+https://github.com/imagdau/aseMolec@f411b5618381ba3b807cbbb041f12f460a69d606

  Cloning https://github.com/imagdau/aseMolec (to revision f411b5618381ba3b807cbbb041f12f460a69d606) to /tmp/pip-req-build-4mfl6cje
  Running command git clone --filter=blob:none --quiet https://github.com/imagdau/aseMolec /tmp/pip-req-build-4mfl6cje
  Running command git rev-parse -q --verify 'sha^f411b5618381ba3b807cbbb041f12f460a69d606'
  Running command git fetch -q https://github.com/imagdau/aseMolec f411b5618381ba3b807cbbb041f12f460a69d606
  Resolved https://github.com/imagdau/aseMolec to commit f411b5618381ba3b807cbbb041f12f460a69d606
  Preparing metadata (setup.py) ... done
  Created wheel for aseMolec: filename=aseMolec-1.0.0-py3-none-any.whl size=23162 sha256=78786bae0181212c3bf933789e9082f07d1c897d7bcabdbe2819f0f90695a83b
  Stored in directory: /root/.cache/pip/wheels/bf/ab/ed/45f9be5430256a5678b55aeccaeb233ab05ac03514a7805562
Successfully built aseMolec


In [ ]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [ ]:
pip install mace-torch cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12 torch-dftd

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.7/266.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.4/215.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.8/197.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.7/681.7 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Import

In [ ]:
import os
from ase.io import write, read
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase.constraints import FixSymmetry
from ase.filters import UnitCellFilter

import time
import numpy as np
import matplotlib.pyplot as plt
from IPython import display

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Otimization


In [ ]:
from mace.calculators import mace_mp

macemp = mace_mp(model="small", dispersion=True, default_dtype = "float32", device="cuda")

/usr/local/lib/python3.12/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


Cached MACE model to /root/.cache/mace/20231210mace128L0_energy_epoch249model
Using Materials Project MACE for MACECalculator with /root/.cache/mace/20231210mace128L0_energy_epoch249model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/usr/local/lib/python3.12/dist-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head Default out of ['Default']
Default dtype float32 does not match model dtype float64, converting models to float32.
Using TorchDFTD3Calculator for D3 dispersion corrections


In [ ]:
atoms = read('/content/drive/MyDrive/HAP.cif')

In [ ]:
atoms.set_constraint(FixSymmetry(atoms))
atoms_filter = UnitCellFilter(atoms)

Definindo o método de otimização e otimizando

In [ ]:
atoms.calc = macemp

from ase.optimize import BFGS

opt = BFGS(atoms_filter)

In [ ]:
print('Initial Energy', atoms.get_potential_energy())
opt.run(fmax=0.001)
print('Final Energy', atoms.get_potential_energy())

/usr/local/lib/python3.12/dist-packages/torch_dftd/torch_dftd3_calculator.py:98: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  cell: Optional[Tensor] = torch.tensor(


Initial Energy -218.47091236465104
      Step     Time          Energy          fmax
BFGS:    0 16:10:20     -218.470912      424.291290
BFGS:    1 16:10:21     -315.693039       20.617674
BFGS:    2 16:10:22     -316.583071       17.179693
BFGS:    3 16:10:22     -318.372117        6.217216
BFGS:    4 16:10:24     -318.802851        3.038035
BFGS:    5 16:10:24     -319.003554        0.681223
BFGS:    6 16:10:24     -319.020825        0.221012
BFGS:    7 16:10:24     -319.026660        0.203994
BFGS:    8 16:10:24     -319.031394        0.183305
BFGS:    9 16:10:24     -319.034183        0.170766
BFGS:   10 16:10:24     -319.037048        0.161254
BFGS:   11 16:10:24     -319.040702        0.153041
BFGS:   12 16:10:24     -319.045054        0.144636
BFGS:   13 16:10:24     -319.048915        0.136600
BFGS:   14 16:10:25     -319.051632        0.129244
BFGS:   15 16:10:25     -319.054192        0.148960
BFGS:   16 16:10:25     -319.058001        0.250842
BFGS:   17 16:10:25     -319.06

Salvando nova estrutura otimizada

In [ ]:
atoms.cell.cellpar()

array([  9.419588  ,   9.419588  ,   6.99139122,  90.        ,
        90.        , 120.        ])

In [ ]:
atoms.set_constraint()
atoms.wrap()
write('/content/drive/MyDrive/HAP_slab.pdb',atoms)
write('/content/drive/MyDrive/HAP_slab.xyz',atoms)
write('/content/drive/MyDrive/HAP_slab.png',atoms)

In [ ]:
atoms = atoms.repeat((2,2,2))
write('/content/drive/MyDrive/HAP_slab_super.pdb',atoms)
write('/content/drive/MyDrive/HAP_slab_super.xyz',atoms)
write('/content/drive/MyDrive/HAP_slab_super.png',atoms)

In [ ]:
atoms.cell.cellpar()

array([ 18.83917601,  18.83917601,  13.98278243,  90.        ,
        90.        , 120.        ])